In [2]:
!pip install gradio numpy opencv-python pillow requests torch torchvision transformers diffusers accelerate


In [3]:
import os
# Model Configuration
DEPTH_MODEL_NAME = "depth-anything/Depth-Anything-V2-Small-hf"
# Stereo Generation Defaults
DEFAULT_STEREO_STRENGTH = 1.00
DEFAULT_MAX_DISPARITY = 70
DEFAULT_INPAINTING_RADIUS = 5
DEFAULT_DEPTH_SMOOTHING = 3
# Supported output formats
OUTPUT_FORMATS = [
    "Side-by-side (SBS)",
    "Half Side-by-side",
    "Over-under (Top/Bottom)",
    "Separate (Left/Right)",
    "Anaglyph (Red/Cyan)"
]



In [4]:
import cv2
import numpy as np
def create_sbs(left_img, right_img, half_width=False):
    """
    Creates a Side-by-Side (SBS) stereoscopic image.
    """
    h, w, c = left_img.shape
    if half_width:
        left_img = cv2.resize(left_img, (w // 2, h))
        right_img = cv2.resize(right_img, (w // 2, h))
    return np.hstack((left_img, right_img))
def create_over_under(left_img, right_img):
    """
    Creates an Over-Under (Top/Bottom) stereoscopic image.
    """
    return np.vstack((left_img, right_img))
def create_anaglyph(left_img, right_img):
    """
    Creates a Red/Cyan Anaglyph stereoscopic image.
    left_img is used for Red channel.
    right_img is used for Green and Blue channels.
    """
    anaglyph = np.zeros_like(left_img)
    # OpenCV uses BGR format
    anaglyph[:, :, 2] = left_img[:, :, 2]   # Red from Left
    anaglyph[:, :, 1] = right_img[:, :, 1]  # Green from Right
    anaglyph[:, :, 0] = right_img[:, :, 0]  # Blue from Right
    return anaglyph
def format_stereo_output(left_img, right_img, output_format):
    """
    Formats the left and right images based on the requested output format.
    """
    if "Separate" in output_format:
        return left_img, right_img
    elif "Half Side-by-side" in output_format:
        return create_sbs(left_img, right_img, half_width=True), None
    elif "Side-by-side" in output_format:
        return create_sbs(left_img, right_img, half_width=False), None
    elif "Over-under" in output_format:
        return create_over_under(left_img, right_img), None
    elif "Anaglyph" in output_format:
        return create_anaglyph(left_img, right_img), None
    else:
        return left_img, right_img                                


In [5]:
import numpy as np
from PIL import Image
import cv2
import torch
from transformers import pipeline
# Global variable to cache the pipeline
_depth_pipeline = None
def get_depth_pipeline():
    global _depth_pipeline
    if _depth_pipeline is None:
        # Determine device
        if torch.cuda.is_available():
            device = 0
            device_type = "cuda"
        elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            device = "mps"
            device_type = "mps"
        else:
            device = -1
            device_type = "cpu"
        print(f"Loading depth estimation pipeline on device: {device_type}")
        _depth_pipeline = pipeline(
            "depth-estimation",
            model=DEPTH_MODEL_NAME,
            device=device,
        )
    return _depth_pipeline
def estimate_depth(image: np.ndarray) -> np.ndarray:
    """
    Estimates the depth of the given image using Depth Anything V2.
    Expects a BGR numpy array (OpenCV format).
    Returns a normalized [0, 1] float32 depth map where
    **1.0 = closest to camera, 0.0 = farthest from camera**.
    """
    h, w = image.shape[:2]

    # Convert BGR → RGB PIL for the HF pipeline
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(image_rgb)
    # Run inference
    pipe = get_depth_pipeline()
    result = pipe(pil_img)
    # ---- Use the raw high-precision tensor instead of the 8-bit PIL image ----
    raw_depth = result["predicted_depth"]           # torch.Tensor [1, H_model, W_model]
    if isinstance(raw_depth, torch.Tensor):
        depth_map = raw_depth.squeeze().cpu().numpy().astype(np.float32)
    else:
        depth_map = np.array(raw_depth, dtype=np.float32)
    # Resize to the original image resolution
    if depth_map.shape[:2] != (h, w):
        depth_map = cv2.resize(depth_map, (w, h), interpolation=cv2.INTER_LINEAR)

    # Normalize to [0, 1]
    d_min, d_max = depth_map.min(), depth_map.max()
    if d_max - d_min > 0:
        depth_map = (depth_map - d_min) / (d_max - d_min)
    else:
        depth_map = np.zeros_like(depth_map)
    # Depth Anything V2 outputs disparity-like values where
    # higher = closer.  Our convention is the same (1 = closest),
    # so no inversion is needed.
    print(f"Depth map stats — min: {depth_map.min():.4f}, max: {depth_map.max():.4f}, "
          f"mean: {depth_map.mean():.4f}")
    return depth_map



In [6]:
import cv2
import numpy as np
from typing import Tuple

def smooth_depth_map(depth_map: np.ndarray, smoothing: int = 5) -> np.ndarray:
    """
    Applies bilateral filtering to smooth depth while preserving edges.
    """
    if smoothing > 0:
        depth_8u = (depth_map * 255).astype(np.uint8)
        smoothed_8u = cv2.bilateralFilter(depth_8u, d=smoothing, sigmaColor=75, sigmaSpace=75)
        return smoothed_8u.astype(np.float32) / 255.0
    return depth_map

def generate_right_eye(
    left_img: np.ndarray,
    depth_map: np.ndarray,
    stereo_strength: float = 0.14,
    max_disparity: int = 70,
    smoothing: int = 3,
    inpaint_radius: int = 5,
    focal_plane: float = 0.0,
    fill_mode: str = "algorithmic",
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Generates a right-eye view by mapping right-image pixels back to the left image.
    Crucially, it detects and rejects "stretched" pixels on steep depth slopes
    (disocclusions), allowing these gaps to be cleanly filled by the true background.
    """
    h, w = left_img.shape[:2]
    # --- 1. Smooth depth & compute disparity ---
    depth_smoothed = smooth_depth_map(depth_map, smoothing)
    # Calculate disparity relative to the focal plane.
    # Positive disp = pops out (shifts left), Negative disp = pushes in (shifts right)
    disp = ((depth_smoothed - focal_plane) * stereo_strength * w).astype(np.float32)
    disp = np.clip(disp, -max_disparity, max_disparity)

    x_map_2d = np.full((h, w), -1, dtype=np.int32)

    # --- 2. Inverse map with stretch rejection (no filling yet) ---
    for y in range(h):
        d = disp[y]
        x_L = np.arange(w)
        # target x_R for each x_L
        x_R = np.round(x_L - d).astype(int)
        # Detect stretched pixels (where depth decreases sharply, i.e. foreground->background)
        # We reject pixels on slopes where disparity drops by > 0.3 px per pixel.
        dd = np.diff(d, append=d[-1])
        valid_L = dd >= -0.3
        # Painter's algorithm: sort by depth (so foreground overwrites background)
        order = np.argsort(d)

        for i in order:
            if not valid_L[i]:
                continue  # Reject stretched slope pixels
            pos = x_R[i]
            if 0 <= pos < w:
                x_map_2d[y, pos] = i

    # --- 3. Dilate gaps to uniformly eat outline pixels ---
    # The dark foreground outline bleeds 3-10px into the background depending
    # on the row. Instead of per-row offsets (inconsistent → streaks), we dilate
    # the gap mask horizontally. This converts ALL outline pixels to gaps on
    # every row uniformly. Then a simple nearest-right fill lands on guaranteed
    # clean background.
    gap_mask = (x_map_2d == -1).astype(np.uint8)
    kernel = np.ones((1, 15), np.uint8)  # 7px dilation each side, horizontal only
    gap_dilated = cv2.dilate(gap_mask, kernel, iterations=1)
    x_map_2d[gap_dilated > 0] = -1

    final_gap_mask = (x_map_2d == -1)

    # --- 4. Fill all gaps from nearest right (background) neighbor if algorithmic ---
    for y in range(h):
            valid_idx = np.where(x_map_2d[y] != -1)[0]
            if len(valid_idx) == 0:
                x_map_2d[y, :] = 0
                continue
            hole_idx = np.where(x_map_2d[y] == -1)[0]
            if len(hole_idx) > 0:
                idx = np.searchsorted(valid_idx, hole_idx)
                idx = np.clip(idx, 0, len(valid_idx) - 1)
                x_map_2d[y, hole_idx] = x_map_2d[y, valid_idx[idx]]

    # --- 5. Fast 2D Color Mapping ---
    y_idx = np.arange(h)[:, None]
    
    # Use np.maximum to prevent index out of bounds during mapping for -1 values
    safe_map = np.maximum(x_map_2d, 0)
    right_img = left_img[y_idx, safe_map]


    return right_img, final_gap_mask


In [7]:
import torch
import numpy as np
from PIL import Image
import cv2
from diffusers import AutoPipelineForInpainting

# Global variable to cache the pipeline
_inpainting_pipeline = None

def get_inpainting_pipeline():
    global _inpainting_pipeline
    if _inpainting_pipeline is None:
        if torch.cuda.is_available():
            device = "cuda"
            dtype = torch.float16
        elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            device = "mps"
            dtype = torch.float32 # mps float16 support can be spotty for some ops
        else:
            device = "cpu"
            dtype = torch.float32

        print(f"Loading inpainting pipeline on device: {device} with dtype: {dtype}")
        _inpainting_pipeline = AutoPipelineForInpainting.from_pretrained(
            "runwayml/stable-diffusion-inpainting",
            torch_dtype=dtype,
            safety_checker=None,
        )
        
        # Offload to CPU if we have CUDA to save VRAM
        if device == "cuda":
            _inpainting_pipeline.enable_model_cpu_offload()
        else:
            _inpainting_pipeline.to(device)

    return _inpainting_pipeline


def inpaint_gaps(image: np.ndarray, gap_mask: np.ndarray) -> np.ndarray:
    if not np.any(gap_mask):
        return image

    h, w = image.shape[:2]

    # Convert BGR -> RGB
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # 1. Resize to max dimension 768 to prevent VAE NaNs (which cause pure black output on large images)
    max_dim = 768
    scale = 1.0
    if max(h, w) > max_dim:
        scale = max_dim / max(h, w)
        new_w = int(w * scale)
        new_h = int(h * scale)
    else:
        new_w, new_h = w, h
        
    resized_img = cv2.resize(image_rgb, (new_w, new_h), interpolation=cv2.INTER_AREA)
    # Mask should be resized with nearest neighbor to keep it strictly boolean/sharp
    resized_mask = cv2.resize(gap_mask.astype(np.uint8), (new_w, new_h), interpolation=cv2.INTER_NEAREST)

    # Pad to multiples of 8
    pad_h = (8 - new_h % 8) % 8
    pad_w = (8 - new_w % 8) % 8
    
    ph, pw = new_h + pad_h, new_w + pad_w
    
    padded_img = cv2.copyMakeBorder(resized_img, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=[0,0,0])
    padded_mask_uint8 = np.pad(resized_mask * 255, ((0, pad_h), (0, pad_w)), mode='constant')

    pil_img = Image.fromarray(padded_img)
    pil_mask = Image.fromarray(padded_mask_uint8)

    pipe = get_inpainting_pipeline()
    
    prompt = "simple clean background, out of focus, seamless"
    negative_prompt = "artifacts, blur, distortion, text, watermark"

    print(f"Running AI Inpainting at {pw}x{ph}...")
    result_img = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=pil_img,
        mask_image=pil_mask,
        height=ph,
        width=pw,
        num_inference_steps=20,
        strength=0.7,
    ).images[0]

    result_np = np.array(result_img)
    
    if result_np.shape[0] != ph or result_np.shape[1] != pw:
        result_np = cv2.resize(result_np, (pw, ph), interpolation=cv2.INTER_LINEAR)
    
    # Crop padding
    result_np = result_np[:new_h, :new_w]
        
    # Resize back to original
    if scale != 1.0:
        result_np = cv2.resize(result_np, (w, h), interpolation=cv2.INTER_LINEAR)
        
    result_bgr = cv2.cvtColor(result_np, cv2.COLOR_RGB2BGR)
    
    final_img = image.copy()
    final_img[gap_mask] = result_bgr[gap_mask]

    return final_img



Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
[transformers] `Siglip2ImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Siglip2ImageProcessor` instead.


In [8]:
import gradio as gr
import cv2
import numpy as np

def create_ui():
    with gr.Blocks(title="2D to 3D Stereoscopic Image Converter") as app:
        gr.Markdown("# 2D to 3D Stereoscopic Image Converter")
        gr.Markdown("Convert any 2D image into a 3D stereoscopic view using local AI depth estimation.")
        with gr.Row():
            with gr.Column():
                input_image = gr.Image(label="Input 2D Image", type="numpy")
                
                with gr.Accordion("Stereo Parameters", open=True):
                    stereo_strength = gr.Slider(0.01, 1.00, value=DEFAULT_STEREO_STRENGTH, step=0.01, label="Stereo Strength", info="Intensity of the 3D effect (how far apart the left/right views are shifted).")
                    max_disparity = gr.Slider(10, 100, value=DEFAULT_MAX_DISPARITY, step=1, label="Maximum Disparity", info="Maximum horizontal shift (in pixels) for the closest foreground objects.")
                    focal_plane = gr.Slider(0.0, 1.0, value=0.0, step=0.05, label="Focal Plane (Zero Parallax)", info="0.0 = Background stays still, foreground pops out. 1.0 = Foreground stays still, background pushes in.")
                    depth_intensity = gr.Slider(0.1, 3.0, value=2.0, step=0.1, label="Depth Intensity (Gamma)", info="Adjusts the non-linear contrast of the depth map. Lower values bring the background closer, higher values push mid-tones deeper.")
                    smoothing = gr.Slider(0, 15, value=DEFAULT_DEPTH_SMOOTHING, step=1, label="Depth Smoothing", info="Bilateral filter size to reduce noise and rough edges on depth boundaries.")
                    
                fill_mode = gr.Radio(choices=["Algorithmic", "AI Inpainting"], value="Algorithmic", label="Background Fill Mode", info="Algorithmic is fast. AI Inpainting is high quality but requires GPU and ~2GB download on first run.")
                output_format = gr.Dropdown(choices=OUTPUT_FORMATS, value="Anaglyph (Red/Cyan)", label="Output Format")
                
                generate_btn = gr.Button("Generate 3D Image", variant="primary")
            
            with gr.Column():
                final_output = gr.Image(label="Final Stereoscopic Output")
                with gr.Accordion("Intermediate Steps", open=False):
                    depth_output = gr.Image(label="Estimated Depth Map")
                    left_output = gr.Image(label="Left Eye View")
                    right_output = gr.Image(label="Right Eye View")
        
        # State to cache depth map
        depth_state = gr.State(None)
        
        def process_image(img, s_strength, m_disp, f_plane, d_intensity, smooth, f_mode, out_fmt, current_depth):
            if img is None:
                return None, None, None, None, current_depth
            
            if current_depth is None:
                try:
                    # img is RGB from Gradio, OpenCV expects BGR internally.
                    img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
                    depth = estimate_depth(img_bgr)
                    current_depth = depth
                except Exception as e:
                    raise gr.Error(f"Depth estimation failed: {str(e)}")
            else:
                img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
            
            # Apply depth intensity (gamma) adjustment
            adjusted_depth = current_depth
            if d_intensity != 1.0:
                adjusted_depth = np.power(current_depth, d_intensity)

            # Generate right eye
            internal_f_mode = "ai" if f_mode == "AI Inpainting" else "algorithmic"
            right_bgr, gap_mask = generate_right_eye(img_bgr, adjusted_depth, s_strength, m_disp, smooth, 5, f_plane, internal_f_mode)
            
            if internal_f_mode == "ai":
                right_bgr = inpaint_gaps(right_bgr, gap_mask)

            # Left eye is original
            left_bgr = img_bgr
            
            # Format output
            out_left, out_right = format_stereo_output(left_bgr, right_bgr, out_fmt)
            
            # If the format is a combined format, out_left contains the combined image and out_right is empty or redundant
            if "Separate" in out_fmt:
                final_bgr = out_left # Just show left in final if separate.
            else:
                final_bgr = out_left
                
            # Convert back to RGB for Gradio
            final_rgb = cv2.cvtColor(final_bgr, cv2.COLOR_BGR2RGB) if len(final_bgr.shape) == 3 else final_bgr
            left_rgb = cv2.cvtColor(left_bgr, cv2.COLOR_BGR2RGB)
            right_rgb = cv2.cvtColor(right_bgr, cv2.COLOR_BGR2RGB)
            
            # Depth for display (0-255 grayscale)
            depth_display = (adjusted_depth * 255).astype(np.uint8)
            
            return final_rgb, depth_display, left_rgb, right_rgb, current_depth

        # When image changes, reset depth state
        input_image.change(
            fn=lambda: None,
            inputs=None,
            outputs=depth_state
        )
        generate_btn.click(
            fn=process_image,
            inputs=[input_image, stereo_strength, max_disparity, focal_plane, depth_intensity, smoothing, fill_mode, output_format, depth_state],
            outputs=[final_output, depth_output, left_output, right_output, depth_state]
        )
    return app
def launch_ui():
    app = create_ui()
    app.launch(debug=True, share=True)
if __name__ == "__main__":
    launch_ui()                    


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://14892576df797ebe04.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://14892576df797ebe04.gradio.live
